In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

seller_mart=pd.read_csv(r"C:\Users\RohanS\Documents\retail project\data retail processed\seller_mart.csv")
df = seller_mart.copy()
df['timedelta'] = pd.to_timedelta(df['avg_delivery'], errors='coerce')
df['total_hours']   = df['timedelta'].dt.total_seconds() / 3600
df["performance_score"] = (
    df["avg_review"].fillna(
        df["avg_review"].median()
    )
    -
    df["total_hours"].fillna(
        df["total_hours"].median()
    ) * 0.1 * np.log(df['total_orders'])
)
df=df.drop(columns='timedelta')
df

,seller_id,total_orders,total_revenue,avg_delivery,avg_review,total_hours,performance_score
0,0015a82c2db000af6aaaf3ae2ecb0532,3,2685.00,10 days 19:03:11.666666666,3.666667,259.053241,-24.793241
1,001cca7ae9ae17fb1caed9dfb1094831,200,25248.93,13 days 01:47:20.500000,3.911765,313.789028,-162.343621
2,001e6ad469a905060d959994f1b41e4f,1,250.00,NaN,1.000000,NaN,1.000000
3,002100f778ceb8431b7a1020ff7ab48f,51,1303.10,15 days 21:26:22.275862069,4.033898,381.439521,-145.941470
4,003554e2dce176b5555353e4f3555ac8,1,120.00,4 days 15:31:24,5.000000,111.523333,5.000000
...,...,...,...,...,...,...,...
3090,ffcfefa19b08742c5d315f2791395ee5,1,69.90,NaN,1.000000,NaN,1.000000
3091,ffdd9f82b9a447f6f8d4b91554cc7dd3,18,2140.80,10 days 06:31:35.190476190,4.285714,246.526442,-66.969592
3092,ffeee66ac5d5a62fe688b9d26f83f534,14,1839.86,18 days 01:47:49.714285714,4.214286,433.797143,-110.267267
3093,fffd5413c0700ac820c7069d66d98c89,60,9360.90,13 days 15:04:06.118644067,3.819672,327.068366,-130.093387


In [26]:


threshold = df[
    "performance_score"
].quantile(0.25)

df["poor_seller"] = (
    df["performance_score"] <= threshold
).astype(int)


print("Poor seller distribution:")
print(
    df["poor_seller"].value_counts()
)



features = [
    "total_orders",
    "total_revenue",
    "total_hours",
    "avg_review"
]


model_df = df[
    features + ["poor_seller"]
].copy()


model_df = model_df.dropna()




X = model_df[features]

y = model_df["poor_seller"]




X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)




model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(
    X_train,
    y_train
)



y_pred = model.predict(
    X_test
)




print("\nAccuracy:")
print(
    accuracy_score(
        y_test,
        y_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)



importance = pd.DataFrame({

    "feature": features,

    "importance": model.feature_importances_

}).sort_values(
    "importance",
    ascending=False
)


print("\nFeature Importance:")
print(importance)




prediction_df = df[
    features
].copy()

prediction_df = prediction_df.fillna(
    prediction_df.median(numeric_only=True)
)


df["poor_seller_probability"] = (
    model.predict_proba(
        prediction_df
    )[:, 1]
)


df["seller_risk"] = pd.cut(

    df["poor_seller_probability"],

    bins=[
        -np.inf,
        0.30,
        0.60,
        np.inf
    ],

    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk"
    ]
)




seller_risk = df[
    [
        "seller_id",
        "total_orders",
        "total_revenue",
        "avg_delivery",
        "avg_review",
        "poor_seller_probability",
        "seller_risk"
    ]
].copy()


seller_risk[
    "poor_seller_probability"
] = seller_risk[
    "poor_seller_probability"
].round(3)


print("\nSeller risk:")
print(
    seller_risk.head(20)
)


print("\nSeller risk distribution:")
print(
    seller_risk[
        "seller_risk"
    ].value_counts()
)


seller_risk.to_csv(
    "C:/Users/RohanS/Desktop/seller_performance_predictions_8_4.csv",
    index=False
)

importance.to_csv(
    "C:/Users/RohanS/Desktop/seller_performance_importance_8_4.csv",
    index=False
)




Poor seller distribution:
poor_seller
0    2321
1     774
Name: count, dtype: int64

Accuracy:
0.9915824915824916

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       439
           1       1.00      0.97      0.98       155

    accuracy                           0.99       594
   macro avg       0.99      0.98      0.99       594
weighted avg       0.99      0.99      0.99       594


Confusion Matrix:
[[439   0]
 [  5 150]]

Feature Importance:
         feature  importance
0   total_orders    0.474420
2    total_hours    0.279123
1  total_revenue    0.206059
3     avg_review    0.040398

Seller risk:
                           seller_id  total_orders  total_revenue  \
0   0015a82c2db000af6aaaf3ae2ecb0532             3        2685.00   
1   001cca7ae9ae17fb1caed9dfb1094831           200       25248.93   
2   001e6ad469a905060d959994f1b41e4f             1         250.00   
3   002100f778ceb8431b7a1020ff7ab48f 